# 🚀 SFT Training on Google Colab

This notebook provides a complete pipeline for Supervised Fine-Tuning (SFT) of Flan-T5 models on Google Colab with GPU acceleration.

## 📋 Table of Contents

### **Setup Phase**
1. [Install Required Packages](#install-packages)
2. [Upload Training Data](#upload-data)
3. [Setup WandB Logging](#setup-wandb)

### **Model & Data Preparation**
4. [Load Base Model](#load-model)
5. [Load and Prepare Training Data](#prepare-data)
6. [Tokenize Dataset](#tokenize-data)

### **Training Phase**
7. [Configure Training Parameters](#training-config)
8. [Start Training](#start-training)

### **Evaluation & Download**
9. [Test Trained Model](#test-model)
10. [Download Model Files](#download-model)

## 🎯 Quick Start
1. **Enable GPU**: Runtime → Change runtime type → GPU
2. **Run cells sequentially** from top to bottom
3. **Upload your `train.json`** when prompted
4. **Monitor training** progress in WandB (optional)
5. **Download trained model** when complete

## 📊 Expected Results
- **Training Time**: 30-60 minutes for 1000 samples
- **Model Size**: ~1.5GB (Flan-T5-base)
- **GPU Memory**: ~8-12GB during training
- **Output**: Trained model files ready for download


## 📦 1. Install Required Packages {#install-packages}

Install all necessary packages for SFT training including transformers, datasets, and WandB for monitoring.


In [1]:
# Install required packages
%pip install transformers datasets accelerate wandb evaluate nltk numpy torch tensorboard

# Verify GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00
CUDA available: True
GPU: Tesla T4
GPU Memory: 15.8 GB


## 📁 2. Upload Training Data {#upload-data}

**Important:** Upload your `train.json` file from the `sft_data/` directory to Colab using the file upload widget below.

### 📋 Data Format Requirements
Your `train.json` file should contain one JSON object per line with the following structure:
```json
{"prompt": "solve: Your question here", "response": "Your detailed answer here"}
```


In [2]:
from google.colab import files
import os

# Create data directory
os.makedirs('sft_data', exist_ok=True)

print("Please upload your train.json file:")
uploaded = files.upload()

# Move uploaded file to correct location
for filename in uploaded.keys():
    if filename.endswith('.json'):
        os.rename(filename, f'sft_data/{filename}')
        print(f"Moved {filename} to sft_data/{filename}")

# Verify data file exists
if os.path.exists('sft_data/train.json'):
    print("✅ Data file uploaded successfully!")
else:
    print("❌ Please upload train.json file")


Please upload your train.json file:


Saving train.json to train.json
Moved train.json to sft_data/train.json
✅ Data file uploaded successfully!


## 📊 3. Setup WandB Logging (Optional) {#setup-wandb}

Configure Weights & Biases for training monitoring and experiment tracking.

### 🔑 WandB Benefits
- **Real-time metrics** visualization
- **Experiment comparison** across runs
- **Model artifact** storage
- **Collaborative** experiment sharing


In [3]:
import wandb

# Login to WandB (optional)
try:
    wandb.login()
    print("✅ WandB login successful!")
    use_wandb = True
except:
    print("⚠️ WandB login failed. Training will continue without logging.")
    use_wandb = False


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tong-zhao (tong-zhao-georgia-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ WandB login successful!


## 🤖 4. Load Base Model {#load-model}

Load the Flan-T5 base model and tokenizer from Hugging Face. This will be our starting point for fine-tuning.

### 📋 Model Information
- **Model**: google/flan-t5-base
- **Parameters**: ~248M
- **Size**: ~1.5GB
- **Task**: Text-to-text generation


In [4]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch

# Load Flan-T5 model and tokenizer from Hugging Face
print("Loading Flan-T5 model and tokenizer...")
model_name = "google/flan-t5-base"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

print(f"✅ Model loaded: {model_name}")
print(f"Model parameters: {model.num_parameters():,}")

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"Model moved to: {device}")


Loading Flan-T5 model and tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ Model loaded: google/flan-t5-base
Model parameters: 247,577,856
Model moved to: cuda


## 📊 5. Load and Prepare Training Data {#prepare-data}

Load your uploaded training dataset and examine the data structure to ensure proper formatting.

### 🔍 Data Validation
This step will:
- Load the JSON dataset
- Display sample data
- Verify data format
- Show dataset statistics


In [13]:
from datasets import load_dataset
import json

# Load the training data
print("Loading training data...")
dataset = load_dataset('json', data_files='sft_data/train.json')

print(f"Dataset loaded: {len(dataset['train'])} samples")
print("Sample data:")
print(json.dumps(dataset['train'][0], indent=2))


Loading training data...
Dataset loaded: 1000 samples
Sample data:
{
  "response": "The distance between two points $(x_1,y_1)$ and $(x_2,y_2)$ in the complex plane is given by the formula $\\sqrt{(x_2-x_1)^2+(y_2-y_1)^2}$.\nIn this case, Joe's point is $(1,2)$ and Gracie's point is $(-1,1)$.\nSo the distance between their points is $\\sqrt{((-1)-(1))^2+((1)-(2))^2}=\\sqrt{(-2)^2+(-1)^2}=\\sqrt{4+1}=\\sqrt{5}$.\nTherefore, Gracie and Joe's points are $\\boxed{\\sqrt{5}}$ units apart.\nThe answer is: \\sqrt{5}",
  "prompt": "solve: Gracie and Joe are choosing numbers on the complex plane. Joe chooses the point $1+2i$. Gracie chooses $-1+i$. How far apart are Gracie and Joe's points?"
}


## 🔤 6. Tokenize Dataset {#tokenize-data}

Convert text data into token IDs that the model can process. This step prepares the data for training.

### ⚙️ Tokenization Process
- **Input tokens**: Convert prompts to token IDs
- **Label tokens**: Convert responses to token IDs  
- **Padding**: Ensure consistent sequence lengths
- **Truncation**: Handle long sequences
- **Debug output**: Verify tokenization quality


In [ ]:
import numpy as np

print("Tokenizing dataset...")

def tokenize_function(examples):
    input_tokens = tokenizer(examples['prompt'], padding="max_length", truncation=True, max_length=512)
    label_tokens = tokenizer(examples['response'], padding="max_length", truncation=True, max_length=512)
    return {
        "input_ids": input_tokens["input_ids"],
        "attention_mask": input_tokens["attention_mask"],
        "labels": label_tokens["input_ids"]
    }

# Use batched=True for efficiency
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset['train'].column_names)

print(f"✅ Training dataset prepared: {len(tokenized_dataset['train'])} samples")
print(f"Sample tokenized data keys: {list(tokenized_dataset['train'][0].keys())}")

# Debug tokenization
sample = tokenized_dataset['train'][0]
print("\n🔍 Debugging tokenization:")
print(f"Original prompt: {dataset['train'][0]['prompt'][:100]}...")  # Fetch original from dataset (tokenized_dataset removed columns)
print(f"Original response: {dataset['train'][0]['response'][:100]}...")
print(f"Input IDs shape: {np.array(sample['input_ids']).shape}")
print(f"Labels shape: {np.array(sample['labels']).shape}")
print(f"Decoded input: {tokenizer.decode(sample['input_ids'][:50])}...")  # First 50 tokens
# Test decoding
# Decode the tensor in the list for inputs
decoded_prompt = tokenizer.decode(sample['input_ids'][0] if isinstance(sample['input_ids'], list) else sample['input_ids'], skip_special_tokens=True)
# Decode the first sample in the labels list
decoded_response = tokenizer.decode(sample['labels'][0], skip_special_tokens=True)
print(f"Decoded prompt: {decoded_prompt[:100]}...")
print(f"Decoded response: {decoded_response[:100]}...")

Tokenizing dataset...


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

✅ Training dataset prepared: 1000 samples
Sample tokenized data keys: ['input_ids', 'attention_mask', 'labels']

🔍 Debugging tokenization:
Original prompt: solve: Gracie and Joe are choosing numbers on the complex plane. Joe chooses the point $1+2i$. Graci...
Original response: The distance between two points $(x_1,y_1)$ and $(x_2,y_2)$ in the complex plane is given by the for...
Input IDs shape: (512,)
Labels shape: (512,)
Decoded input: solve: Gracie and Joe are choosing numbers on the complex plane. Joe chooses the point $1+2i$. Gracie chooses $-1+i$. How far apart are Gracie and Joe's points?...
Decoded prompt: solve...
Decoded response: The...


## ⚙️ 7. Configure Training Parameters {#training-config}

Set up training configuration optimized for Colab GPU environment.

### 🔧 Training Configuration
- **Epochs**: 3 (adjustable)
- **Batch Size**: 4 (optimized for T4 GPU)
- **Learning Rate**: 1e-4
- **Mixed Precision**: FP16 enabled
- **Gradient Accumulation**: 4 steps
- **Data Collator**: Seq2Seq for proper padding


## 🚀 8. Start Training {#start-training}

Begin the fine-tuning process. This will take 30-60 minutes depending on your dataset size.

### 📊 Training Process
- **Initialize Trainer** with model, data, and configuration
- **Start training** with progress monitoring
- **Save model** automatically when complete
- **Download model** as zip file


In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

# Data collator for seq2seq tasks (handles padding, labels=-100 for ignored tokens)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Training arguments
training_args = TrainingArguments(
    output_dir="./sft_results",
    num_train_epochs=3,  # Adjust as needed
    per_device_train_batch_size=4,  # Adjust based on GPU memory (T4 can handle ~8-16 for flan-t5-base)
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    fp16=True,  # Mixed precision for GPU efficiency
    save_steps=500,
    logging_steps=100,
    report_to="wandb" if use_wandb else "none",  # Integrate WandB if logged in
    load_best_model_at_end=False,
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("Starting training...")
trainer.train()

# Save the model
trainer.save_model("./sft_trained_model")
tokenizer.save_pretrained("./sft_trained_model")
print("✅ Model saved to ./sft_trained_model")

# Download the model (Colab-specific)
from google.colab import files
import shutil
shutil.make_archive("sft_trained_model", 'zip', "./sft_trained_model")
files.download("sft_trained_model.zip")

/tmp/ipython-input-843864834.py:21: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting training...


Step,Training Loss
100,0.000000


✅ Model saved to ./sft_trained_model


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 🧪 9. Test Trained Model {#test-model}

Evaluate your fine-tuned model with sample prompts to verify training quality.

### 🔍 Testing Process
- **Multiple test prompts** for comprehensive evaluation
- **Beam search** and **greedy decoding** comparison
- **Quality assessment** of generated responses
- **Performance validation** before deployment


In [27]:
# Test the trained model (Improved version)
print("🧪 Testing trained model...")

test_prompts = [
    "solve: What is 2 + 2?",
    "solve: If a train travels 60 miles in 1 hour, how far will it travel in 3 hours?",
    "solve: Calculate the area of a circle with radius 5."
]

model.eval()
with torch.no_grad():
    for i, prompt in enumerate(test_prompts):
        print(f"\n--- Test {i+1} ---")
        print(f"Prompt: {prompt}")

        # Tokenize input
        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        # Generate response with better parameters
        outputs = model.generate(
            inputs.input_ids,
            max_length=256,
            num_beams=4,
            early_stopping=True,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1
        )

        # Decode response
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f"Model response: {response}")

        # Also test greedy decoding
        outputs_greedy = model.generate(
            inputs.input_ids,
            max_length=256,
            do_sample=False,
            num_beams=1
        )
        response_greedy = tokenizer.decode(outputs_greedy[0], skip_special_tokens=True)
        print(f"Greedy response: {response_greedy}")

print("\n🎉 Model testing completed!")
print("If responses look poor, the model may need more training or different hyperparameters.")


🧪 Testing trained model...

--- Test 1 ---
Prompt: solve: What is 2 + 2?
Model response: two
Greedy response: a doubling of two

--- Test 2 ---
Prompt: solve: If a train travels 60 miles in 1 hour, how far will it travel in 3 hours?
Model response: 60 miles / hour * 1 hour = 60 miles / hour. 3 hours = 60 miles / hour * 3 hours = 180 miles / hour. The answer: 180.
Greedy response: 60 miles / hour = 240 miles / hour. 3 hours = 3 hours. 240 miles / hour = 240 miles / hour. 240 miles / hour = 240 miles / hour. 240 miles / hour = 240 miles / hour. 240 miles / hour = 240 miles / hour. 240 miles / hour = 240 miles / hour. 240 miles / hour = 240 miles / hour. 240 miles / hour = 240 miles / hour. 240 miles / hour = 240 miles / hour. 240 miles / hour = 240 miles / hour. 240 miles / hour = 240 miles / hour. 240 miles / hour = 240 miles / hour. 240 miles / hour = 240 miles / hour. 240 miles / hour = 240 miles / hour. 240 miles / hour = 240 miles / hour. 240 miles / hour = 240 miles / hour. 240 mil